# Week 2 · Module 1 Practical — Your First RAG Pipeline 🔍

**PMS RoBoTics Research Center · 4-Week Internship & Training Program**
**Week 2 · Module 1: Why LLMs hallucinate and how RAG solves it with external knowledge**

---

### What you'll do in the next 30 minutes

| # | Experiment | Skill |
|---|-----------|-------|
| 1 | Catch the failure | Prove your Week 1 bot can't do catalog questions |
| 2 | Measure the naive fix | Context stuffing: works, but count the tokens |
| 3 | Build a retriever | **BM25** keyword search over the catalog |
| 4 | Wire the RAG loop | retrieve → augment → generate, with citations |
| 5 | Score the retriever | **Precision@k · Recall@k · F1 · MRR@K** by hand |
| 🏁 | **RAGBot v0.1** | Grounded answers + a retrieval scorecard |

> 📏 **New this week:** every build ends with numbers. "It seems to work" is Week 1 talk.

## Step 0 — Setup 🔑

New library: `rank_bm25` — a tiny, pure-Python implementation of the **BM25** ranking algorithm (the keyword-search workhorse behind classic search engines).

In [ ]:
%pip install -q -U openai rank_bm25 tiktoken

In [ ]:
from getpass import getpass
from openai import OpenAI
import tiktoken

API_KEY = getpass("Paste the OpenAI API key: ")
client = OpenAI(api_key=API_KEY)
MODEL = "gpt-4o-mini"
enc = tiktoken.get_encoding("o200k_base")

def ask(messages, temperature=0.3, max_tokens=250):
    r = client.chat.completions.create(model=MODEL, messages=messages,
                                       temperature=temperature, max_tokens=max_tokens)
    return r.choices[0].message.content.strip(), r.usage

print("✅ Ready. Model:", MODEL)

### The knowledge base: SmartMart's catalog 📦

20 products + 4 policy documents. Notice the **chunking decision already made for you**: one product = one chunk, each carrying its own ID, category and full context (that's *context-aware chunking by construction* — tomorrow we do it to messy real documents).

In [ ]:
CATALOG = [
    "P-101 | Prima Electric Kettle 1.5L | Rs. 1,299 | kitchen | 1500W fast boil, auto shut-off, 1-year warranty | in stock: 42",
    "P-102 | Prima Electric Kettle 2.0L | Rs. 1,699 | kitchen | family size, keep-warm mode, 1-year warranty | in stock: 18",
    "P-103 | Zenith Steel Kettle 1.0L | Rs. 899 | kitchen | budget stovetop kettle, no electricity needed | in stock: 65",
    "P-104 | Zenith Pro Kettle 1.8L | Rs. 2,499 | kitchen | premium, temperature control, 2-year warranty | in stock: 7",
    "P-105 | SwiftMix Mixer Grinder 750W | Rs. 3,499 | kitchen | 3 jars, overload protection, 2-year warranty | in stock: 23",
    "P-106 | SwiftMix Mixer Grinder 500W | Rs. 2,299 | kitchen | 2 jars, compact, 1-year warranty | in stock: 31",
    "P-107 | AirChef Air Fryer 4L | Rs. 5,999 | kitchen | oil-free frying, digital timer, 1-year warranty | in stock: 12",
    "P-108 | AirChef Air Fryer 6L XL | Rs. 8,499 | kitchen | family size, 8 presets, 2-year warranty | in stock: 4",
    "P-109 | CoolBreeze Table Fan 400mm | Rs. 1,899 | appliances | 3-speed, oscillating, 2-year warranty | in stock: 55",
    "P-110 | CoolBreeze Tower Fan | Rs. 4,299 | appliances | remote control, timer, quiet mode | in stock: 9",
    "P-111 | BrightHome LED Bulb 9W (pack of 4) | Rs. 499 | electrical | warm white, 2-year replacement | in stock: 210",
    "P-112 | BrightHome Smart Bulb WiFi | Rs. 799 | electrical | app control, 16M colors, voice assistant support | in stock: 48",
    "P-113 | PowerSafe Extension Board 6-socket | Rs. 649 | electrical | surge protection, 2m cord | in stock: 87",
    "P-114 | SmartWatch Fit Pro | Rs. 2,999 | gadgets | heart rate, SpO2, 7-day battery | in stock: 26",
    "P-115 | EarBuds Bass+ TWS | Rs. 1,499 | gadgets | 24h playback, IPX4, ENC mic | in stock: 39",
    "P-116 | PixelSnap Barcode Scanner X-500 | Rs. 4,999 | business | wireless, 18h battery, 1-year warranty | in stock: 6",
    "P-117 | ThermoSteel Water Bottle 1L | Rs. 749 | home | 12h hot / 24h cold, leak-proof | in stock: 120",
    "P-118 | CleanSweep Robot Vacuum | Rs. 12,999 | home | app control, auto-dock, HEPA filter | in stock: 3",
    "P-119 | FreshBrew Coffee Maker 600ml | Rs. 2,799 | kitchen | drip brew, keep-warm plate, 1-year warranty | in stock: 14",
    "P-120 | ChopMaster Vegetable Chopper | Rs. 599 | kitchen | 900ml, 3 blades, dishwasher safe | in stock: 73",
    "D-201 | POLICY delivery | Free delivery above Rs. 999, otherwise Rs. 49. Standard 2-4 days in Pune, 4-7 days rest of Maharashtra.",
    "D-202 | POLICY returns | Returns within 7 days with receipt for full refund. Without receipt: store credit only. Electronics must be unopened.",
    "D-203 | POLICY warranty | Warranty claims need invoice + product serial number. Service center: SmartMart Pimple Saudagar, 9 AM-9 PM.",
    "D-204 | POLICY offers | Current offer: 10% off all kitchen appliances till Sunday. Not combinable with other coupons.",
]
print(f"Knowledge base: {len(CATALOG)} chunks")
print(f"Total size: {sum(len(enc.encode(c)) for c in CATALOG)} tokens")

---
## Experiment 1 — Catch the failure 🕵️

Week 1's bot, unchanged. Watch it face catalog questions:

In [ ]:
WEEK1_SYSTEM = """You are SmartBot, the customer assistant of SmartMart retail store, Pune.
Store hours: 9 AM - 9 PM. Returns: 7 days with receipt. Delivery: free above Rs. 999.
Answer in max 3 sentences. If you don't know something, say
"I'll need to check our system for that." Never invent prices or stock."""

questions = [
    "What's the price of the Prima 1.5L kettle?",
    "Which is your cheapest kettle?",
    "Is the CleanSweep robot vacuum in stock?",
]
for q in questions:
    answer, _ = ask([{"role": "system", "content": WEEK1_SYSTEM},
                     {"role": "user", "content": q}])
    print(f"🧑 {q}")
    print(f"🛒 {answer}")
    print("-" * 70)

**Best case:** it deflects every time (your Week 1 constraints working). **Worst case:** remove the 'never invent' rule and re-run — watch it hallucinate a price with total confidence. Either way, the customer got no answer. The knowledge exists — it's just not *reachable*.

---
## Experiment 2 — The naive fix, measured 📏

Paste the *entire* catalog into the prompt. It works! Now look at the bill:

In [ ]:
full_context = "\n".join(CATALOG)
stuffed_system = WEEK1_SYSTEM + "\n\nFULL CATALOG:\n" + full_context

answer, usage = ask([{"role": "system", "content": stuffed_system},
                     {"role": "user", "content": "Which is your cheapest kettle?"}])
print("🛒", answer)
print(f"\n📊 input tokens this call: {usage.prompt_tokens}")

# Now scale it like a real store:
per_chunk = sum(len(enc.encode(c)) for c in CATALOG) / len(CATALOG)
for n_products in [24, 500, 5000]:
    tokens = int(per_chunk * n_products)
    print(f"{n_products:>5} products → ~{tokens:>7,} tokens EVERY call"
          + ("   ⟵ our toy catalog" if n_products == 24 else "")
          + ("   ⟵ exceeds many context windows!" if n_products == 5000 else ""))

**It works at 24 chunks and dies at 5,000.** Cost scales with catalog size instead of question complexity, and long contexts bury facts ('lost in the middle').

**What we want:** fetch only the 3 chunks that matter for *this* question. That's a **retriever**.

---
## Experiment 3 — Build a BM25 retriever 🔎

**BM25** ranks documents for a query using: **term frequency** (your words appear often) × **IDF** (rare words like 'kettle' count more than 'the') with **length normalization** (long docs can't cheat). No ML, no API — pure math from the 90s that still powers serious search:

In [ ]:
from rank_bm25 import BM25Okapi

def tokenize(text):
    return text.lower().replace("|", " ").replace(",", " ").split()

bm25 = BM25Okapi([tokenize(c) for c in CATALOG])

def retrieve(query, k=3):
    """Return the top-k chunks for a query, with scores."""
    scores = bm25.get_scores(tokenize(query))
    ranked = sorted(range(len(CATALOG)), key=lambda i: scores[i], reverse=True)[:k]
    return [(CATALOG[i], scores[i]) for i in ranked]

for chunk, score in retrieve("cheapest electric kettle price", k=3):
    print(f"score {score:5.2f} | {chunk[:80]}...")

In [ ]:
# Poke at its strengths and weaknesses:
test_queries = [
    "robot vacuum stock",            # exact keywords -> should nail it
    "X-500 scanner warranty",        # model numbers -> BM25's superpower
    "something affordable for boiling water",   # synonyms -> struggles?
    "birthday gift under 1000 rupees",           # concepts -> struggles?
]
for q in test_queries:
    top = retrieve(q, k=1)[0]
    print(f"Q: {q}")
    print(f"→ {top[0][:75]}...  (score {top[1]:.2f})")
    print()

**Observe:** exact keywords and model numbers → excellent. Synonyms and concepts ('affordable' vs 'budget', 'boiling water' vs 'kettle') → shaky. Keep these failure queries — they're **tomorrow's motivation for embeddings**, and Day 3's case for **hybrid search (BM25 + vectors via HNSW)**.

---
## Experiment 4 — Wire the RAG loop 🔗

Retrieve → Augment → Generate. The grounding template does the safety work:

In [ ]:
RAG_TEMPLATE = """You are SmartBot, the customer assistant of SmartMart retail store, Pune.

Answer ONLY from the context below. If the answer is not in the context,
say "I don't have that information — let me connect you with our team."
Cite the product/policy ID you used, like [P-101].
Answer in max 3 sentences, warm and professional.

CONTEXT:
{context}

CUSTOMER QUESTION: {question}"""

def rag_answer(question, k=3, verbose=False):
    chunks = retrieve(question, k=k)
    context = "\n".join(c for c, _ in chunks)
    if verbose:
        print("── retrieved ──")
        for c, s in chunks:
            print(f"  [{s:5.2f}] {c[:70]}...")
        print("───────────────")
    answer, usage = ask([{"role": "user",
                          "content": RAG_TEMPLATE.format(context=context, question=question)}])
    return answer, usage

answer, usage = rag_answer("Which is your cheapest kettle?", verbose=True)
print("🛒", answer)
print(f"📊 input tokens: {usage.prompt_tokens}  (vs {len(enc.encode(full_context))+120} stuffed!)")

In [ ]:
# The questions that broke Week 1's bot — plus a trap:
for q in ["What's the price of the Prima 1.5L kettle?",
          "Is the CleanSweep robot vacuum in stock?",
          "What's the warranty on the X-500 barcode scanner?",
          "Do you sell washing machines?"]:          # NOT in catalog — must say IDK
    answer, _ = rag_answer(q)
    print(f"🧑 {q}")
    print(f"🛒 {answer}")
    print("-" * 70)

🎉 **Real prices, real stock, citations, and an honest 'I don't have that' for washing machines.** Same LLM as Friday — only its reading material changed.

**✏️ Your turn:** ask something answered by a POLICY chunk ("can I return an opened air fryer?") and check the citation.

---
## Experiment 5 — Score the retriever 📐

*"It seems to work"* ends now. We build a **test set** — queries with known relevant chunks — and compute the four retrieval metrics:

- **Precision@k** — of the k chunks fetched, what fraction were relevant? *(junk check)*
- **Recall@k** — of all relevant chunks, what fraction did we fetch? *(missed check)*
- **F1** — harmonic mean of the two
- **MRR@K** — 1/rank of the first relevant hit, averaged *(is the right answer at the top?)*

In [ ]:
# Gold test set: query -> IDs of chunks that ARE relevant (judged by us, the humans)
TEST_SET = [
    ("price of prima 1.5L kettle",              {"P-101"}),
    ("cheapest kettle",                          {"P-103"}),
    ("robot vacuum in stock?",                   {"P-118"}),
    ("X-500 scanner warranty",                   {"P-116", "D-203"}),
    ("return policy without receipt",            {"D-202"}),
    ("delivery charge for Rs. 500 order",        {"D-201"}),
    ("discount on kitchen items",                {"D-204"}),
    ("mixer grinder with overload protection",   {"P-105"}),
    ("affordable way to boil water",             {"P-103", "P-101"}),   # synonym query!
    ("smart lighting for home",                  {"P-112"}),
]

def chunk_id(chunk):
    return chunk.split("|")[0].strip().split()[0]

def evaluate(k=3):
    precisions, recalls, f1s, rrs = [], [], [], []
    print(f"{'query':<42} P@{k}   R@{k}   RR")
    for query, relevant in TEST_SET:
        retrieved_ids = [chunk_id(c) for c, _ in retrieve(query, k=k)]
        hits = [rid for rid in retrieved_ids if rid in relevant]
        p = len(hits) / k
        r = len(set(hits)) / len(relevant)
        f1 = 2 * p * r / (p + r) if (p + r) else 0.0
        rr = 0.0
        for rank, rid in enumerate(retrieved_ids, start=1):
            if rid in relevant:
                rr = 1.0 / rank
                break
        precisions.append(p); recalls.append(r); f1s.append(f1); rrs.append(rr)
        print(f"{query:<42} {p:.2f}  {r:.2f}  {rr:.2f}")
    n = len(TEST_SET)
    print("-" * 62)
    print(f"{'MEAN':<42} {sum(precisions)/n:.2f}  {sum(recalls)/n:.2f}")
    print(f"F1 (mean): {sum(f1s)/n:.2f}   |   MRR@{k}: {sum(rrs)/n:.2f}")
    return sum(f1s)/n, sum(rrs)/n

f1_at_3, mrr_at_3 = evaluate(k=3)

In [ ]:
# Engineering question #1: what's the right k? Measure, don't guess:
print("k | mean F1 | MRR@k")
for k in [1, 3, 5, 10]:
    import io, contextlib
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):     # silence the per-query table
        f1, mrr = evaluate(k=k)
    print(f"{k:>2} |  {f1:.2f}   | {mrr:.2f}")

**Read your table like an engineer:**
- Small k → precision up, recall down (you fetch less junk but miss things)
- Big k → recall up, precision down (and more tokens per call — cost!)
- MRR tells you whether the *first* hit is usually right — critical when you only show one answer

**Which queries scored worst?** Almost certainly the synonym ones — BM25's known blind spot. Tomorrow embeddings fix exactly that, *and we'll re-run this same test set to prove the improvement with numbers.* On Day 4, **RAGAS** adds LLM-graded metrics (faithfulness, answer relevancy, context precision/recall) for the generation side.

---
## 🏁 Finale — RAGBot v0.1

The full loop in one class — Week 1's plumbing + today's retrieval:

In [ ]:
class RAGBot:
    """SmartBot + BM25 retrieval. v0.1 — keyword baseline."""

    def __init__(self, k=3):
        self.k = k
        self.total_in = 0
        self.total_out = 0

    def say(self, question, verbose=False):
        answer, usage = rag_answer(question, k=self.k, verbose=verbose)
        if usage:
            self.total_in += usage.prompt_tokens
            self.total_out += usage.completion_tokens
        return answer

    def scorecard(self):
        f1, mrr = None, None
        import io, contextlib
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf):
            f1, mrr = evaluate(k=self.k)
        return {"k": self.k, "mean_F1": round(f1, 2), "MRR": round(mrr, 2),
                "tokens_in": self.total_in, "tokens_out": self.total_out}

bot = RAGBot(k=3)
print("🛒", bot.say("I want something to make coffee at home — what do you have and what does it cost?"))
print("🛒", bot.say("Is it covered by warranty and how do I claim it?"))
print()
print("📊 Scorecard:", bot.scorecard())

In [ ]:
# 💬 Chat with RAGBot — type 'quit' to stop, 'v' before a question for verbose retrieval view.
while True:
    user_msg = input("You: ")
    if user_msg.strip().lower() in ("quit", "exit", "q"):
        print("👋 Scorecard:", bot.scorecard())
        break
    verbose = user_msg.startswith("v ")
    print("🛒", bot.say(user_msg[2:] if verbose else user_msg, verbose=verbose))

---
## 🏆 Homework (15 min, feeds the Week 2 mini-project)

1. **Break BM25, with evidence** — add 3 test queries where keyword search fails (synonyms, paraphrases, or Hinglish: *"sasta kettle dikhao"*). Re-run `evaluate(k=3)` and note how far F1 and MRR fall. Bring the numbers tomorrow.
2. **Tune k** — using the k-sweep table, pick the k YOU would ship and defend it in 2 sentences (hint: mention both MRR and token cost).
3. **One missing chunk** — add a new product to CATALOG (keep the format!), add one test query for it, re-index (`bm25 = BM25Okapi(...)`), and confirm retrieval finds it. You just did a knowledge-base refresh — no retraining, no redeploy.

### What's next
**Module 2 — Embeddings & context-aware chunking:** how machines represent *meaning* as vectors (fixing every synonym failure in your homework), and how to chunk messy real documents so each piece keeps its context.

---
*PMS RoBoTics Research Center · pmsroboticsrc@gmail.com · (+91) 9960664674*